In [1]:
import sys
sys.path.insert(0, "..")
import mlflow.tensorflow
import tensorflow as tf
from loguru import logger

from commons.constants import (
    MODEL_NAME,
    STORAGE_PATH,
)
from commons.mlflow_tools import connect_remote_mlflow
from two_towers_model.utils.evaluate import evaluate


STORAGE_PATH = "gs://mlflow-bucket-prod/algo_training_prod/algo_training_two_towers_20250227T132939"
experiment_name = "algo_training_two_towers_v1.2_prod"
run_id = "e6e14101ec2b463296fe7b1182375d6c"
model_name=MODEL_NAME
training_dataset_name = "recommendation_training_data"
test_dataset_name="recommendation_test_data"

logger.info("-------EVALUATE START------- ")

connect_remote_mlflow()
experiment_id = mlflow.get_experiment_by_name(experiment_name).experiment_id
with mlflow.start_run(experiment_id=experiment_id, run_id=run_id) as run:
    artifact_uri = mlflow.get_artifact_uri("model")
    loaded_model = tf.keras.models.load_model(
        artifact_uri,
        compile=False,
    )


metrics = evaluate(
            model=loaded_model,
            storage_path=STORAGE_PATH,
            training_dataset_name=training_dataset_name,
            test_dataset_name=test_dataset_name,
            list_k=[10, 40],
            all_users=False,
            dummy=True, 
            quantile_threshold=0.90
        )

#2^(novelty) is the average percentage of items in total intercations
metrics

2025-03-11 17:11:33.559 | INFO     | __main__:<module>:22 - -------EVALUATE START------- 
2025-03-11 17:12:07.527 | INFO     | two_towers_model.utils.evaluate:evaluate:229 - Get data for evaluation
2025-03-11 17:12:07.527 | INFO     | two_towers_model.utils.evaluate:load_data_for_evaluation:43 - Load training


🏃 View run updated-loss-202502 at: https://mlflow.passculture.team/#/experiments/35/runs/e6e14101ec2b463296fe7b1182375d6c
🧪 View experiment at: https://mlflow.passculture.team/#/experiments/35


2025-03-11 17:13:18.359 | INFO     | two_towers_model.utils.evaluate:load_data_for_evaluation:47 - Number of unique (user, item) interactions in training data: 13034735
2025-03-11 17:13:18.361 | INFO     | two_towers_model.utils.evaluate:load_data_for_evaluation:51 - Load test data
2025-03-11 17:13:22.948 | INFO     | two_towers_model.utils.evaluate:load_data_for_evaluation:57 - Number of unique (user, item, ) interactions in test data: 609724
2025-03-11 17:13:22.949 | INFO     | two_towers_model.utils.evaluate:evaluate:236 - Getting predictions
2025-03-11 17:13:23.286 | INFO     | two_towers_model.utils.evaluate:generate_predictions:93 - Computing metrics for 40 users
100%|██████████| 40/40 [04:08<00:00,  6.21s/it]
2025-03-11 17:17:31.833 | INFO     | two_towers_model.utils.evaluate:evaluate:241 - Computing model metrics
/Users/Carole/projects/data-gcp/jobs/ml_jobs/algo_training/.venv/lib/python3.10/site-packages/recommenders/evaluation/python_evaluation.py:1435: FutureWarning: Series

{'precision@10': 0.0225,
 'recall@10': 0.1875,
 'novelty@10': 14.090096713849855,
 'coverage@10': 0.17304174071370193,
 'precision@40': 0.008750000000000003,
 'recall@40': 0.3,
 'novelty@40': 14.090096713849855,
 'coverage@40': 0.17304174071370193,
 'random_precision@10': 0.0,
 'random_recall@10': 0.0,
 'random_novelty@10': 14.090096713849855,
 'random_coverage@10': 0.17304174071370193,
 'popular_precision@10': 0.0,
 'popular_recall@10': 0.0,
 'popular_novelty@10': 14.090096713849855,
 'popular_coverage@10': 0.17304174071370193,
 'random_precision@40': 0.0,
 'random_recall@40': 0.0,
 'random_novelty@40': 14.090096713849855,
 'random_coverage@40': 0.17304174071370193,
 'popular_precision@40': 0.0,
 'popular_recall@40': 0.0,
 'popular_novelty@40': 14.090096713849855,
 'popular_coverage@40': 0.17304174071370193}

In [4]:
from two_towers_model.utils.evaluate import prepare_data_for_evaluation, generate_predictions, compute_metrics, generate_random_baseline, generate_popularity_baseline
model=loaded_model
storage_path=STORAGE_PATH
training_dataset_name = "recommendation_training_data"
test_dataset_name = "recommendation_test_data"
list_k = [10, 40]
dummy = False

test_data, training_data, users_to_test = prepare_data_for_evaluation(
    storage_path=storage_path,
    training_dataset_name=training_dataset_name,
    test_dataset_name=test_dataset_name,
    n_users_to_test=10,  # EVALUATION_USER_NUMBER,
)

logger.info("Get predictions")
df_predictions = generate_predictions(loaded_model, test_data, users_to_test)

# logger.info("Compute metrics")
# metrics = {}
# for k in list_k:
#     metrics.update(compute_metrics(test_data, df_predictions, training_data, k=k))

# if dummy:
#     logger.info("Generating random and popularity baselines")
#     df_random = generate_random_baseline(
#         test_data, users_to_test, num_recommendations=10
#     )
#     df_popular = generate_popularity_baseline(
#         training_data, test_data, users_to_test, num_recommendations=10
#     )

#     for k in list_k:
#         metrics.update(
#             compute_metrics(
#                 test_data, df_random, training_data, k=k, prefix="random_"
#             )
#         )
#         metrics.update(
#             compute_metrics(
#                 test_data, df_popular, training_data, k=k, prefix="popular_"
#             )
#         )



2025-03-11 15:39:24.377 | INFO     | two_towers_model.utils.evaluate:prepare_data_for_evaluation:49 - Load training
2025-03-11 15:40:43.449 | INFO     | two_towers_model.utils.evaluate:prepare_data_for_evaluation:53 - Number of unique (user, item, offer_subcategory_id) interactions in training data: 13034735
2025-03-11 15:40:43.451 | INFO     | two_towers_model.utils.evaluate:prepare_data_for_evaluation:57 - Load test data
2025-03-11 15:40:48.752 | INFO     | two_towers_model.utils.evaluate:prepare_data_for_evaluation:65 - Number of unique (user, item, offer_subcategory_id) interactions in test data: 609724
2025-03-11 15:40:48.991 | INFO     | two_towers_model.utils.evaluate:prepare_data_for_evaluation:75 - Computing metrics for 10 users
2025-03-11 15:40:48.992 | INFO     | __main__:<module>:16 - Get predictions
  0%|          | 0/10 [00:00<?, ?it/s]/Users/Carole/projects/data-gcp/jobs/ml_jobs/algo_training/two_towers_model/../two_towers_model/utils/evaluate.py:136: FutureWarning: The 

In [9]:
test_data.item_id.nunique(), test_data[['item_id', 'offer_subcategory_id']].drop_duplicates().item_id.nunique()

(118988, 118988)

In [10]:
max([10,50])

50

In [6]:
import pandas as pd
pd.merge(df_predictions, training_data, on=["user_id", "item_id"], how="inner")

,user_id,item_id,score,offer_subcategory_id
0,2002835,link-482529e6-364e-4a4e-97d5-f45cef331088,0.594540,CONCERT
1,2002835,link-47f5998a-9fb4-4702-94fb-5d4d90909f5d,0.485185,CARTE_CINE_MULTISEANCES
2,2002835,product-6269249,0.511653,SUPPORT_PHYSIQUE_MUSIQUE_CD
3,2002835,link-7234b7d4-15db-4dc3-bd83-a74cba1c09de,0.468589,SPECTACLE_REPRESENTATION
4,2002835,link-0b1021df-d804-4d7b-96f1-55a83ff2b58b,0.662875,CONCERT
...,...,...,...,...
97,3869042,product-18208,0.564355,LIVRE_PAPIER
98,3869042,product-2266603,0.536753,LIVRE_PAPIER
99,3869042,product-6291067,0.210017,SUPPORT_PHYSIQUE_MUSIQUE_CD
100,3869042,product-1877647,0.473953,LIVRE_PAPIER
